In [1]:
# Cell 1 - Load dataset
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
train_set = dataset["train"]
test_set = dataset["test"]


In [ ]:
# Cell 2 - Find similar examples for 3-shot
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def get_similarity(a, b):
    emb = embedder.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

similarities = []
l = len(train_set)
for i in range(l):
    for j in range(i+1, l):
        s = get_similarity(train_set["question"][i], train_set["question"][j])
        similarities.append((i, j, s))
        print(i, j)

KeyboardInterrupt: 

In [ ]:
# Cell 3 - Pick 3 similar examples
sorted_similarities = sorted(similarities, key=lambda x: x[2], reverse=True)

visited_pairs = []
for i, j, s in sorted_similarities:
    for pair in visited_pairs:
        if i in pair or j in pair:
            visited_pairs.append((i, j, s))
            break
    else:
        visited_pairs.append((i, j, s))
print(visited_pairs)
print()
examples_3shot = [
    (train_set["question"][3],  train_set["answer"][3]),
    (train_set["question"][14], train_set["answer"][14]),
    (train_set["question"][94], train_set["answer"][94])
]
examples_3shot

In [ ]:
# Cell 4 - Load model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"# Larger model for bigger accuracy

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda"
)

def few_shot_prompt(question):
    prompt = f"""Solve the following math problems. Always end your answer with #### <number> where <number> is the final answer.

Q: {examples_3shot[0][0]}
A: {examples_3shot[0][1]}

Q: {examples_3shot[1][0]}
A: {examples_3shot[1][1]}

Q: {examples_3shot[2][0]}
A: {examples_3shot[2][1]}

Q: {question}
A:"""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # ← must not be commented out

    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.3,
        top_p=0.95
    )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = output[0][input_length:]  # ← only decode new tokens
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [ ]:
# Cell 5 - Evaluate - same qn 100 times
import re

def extract_correct_answer(text):
    match = re.search(r'#### (\d+)', text)
    return match.group(1) if match else None

def extract_answer(text):
    # Try #### format first
    match = re.search(r'#### (\d+)', text)
    if match:
        return match.group(1)
    
    # Try **$195** or **195** (bold markdown format)
    match = re.search(r'\$(\d+)', text)
    if match:
        return match.group(1)
    
    # Fallback: last number in text
    matches = re.findall(r'\d+', text)
    return matches[-1] if matches else None

correct_count = 0
correct_answer = extract_correct_answer(test_set["answer"][200])

print("Q: ", test_set["question"][200])
print("Correct answer:", correct_answer)

for i in range(100):
    print(f"RUN #{i+1}")
    text = few_shot_prompt(test_set["question"][200])
    print(text)
    answer = extract_answer(text)
    print(f"  Model: {answer} | Expected: {correct_answer}")
    if answer == correct_answer:
        correct_count += 1

print("Final Correct count:", correct_count)
print("Accuracy:", correct_count / 100)

In [ ]:
# Cell 6 - Evaluate - 100 different qns
import re

def extract_correct_answer(text):
    match = re.search(r'#### (\d+)', text)
    return match.group(1) if match else None

def extract_answer(text):
    # Try #### format first
    match = re.search(r'#### (\d+)', text)
    if match:
        return match.group(1)
    
    # Try **$195** or **195** (bold markdown format)
    match = re.search(r'\$(\d+)', text)
    if match:
        return match.group(1)
    
    # Fallback: last number in text
    matches = re.findall(r'\d+', text)
    return matches[-1] if matches else None

correct_count = 0
correct_answer = extract_correct_answer(test_set["answer"][200])

print("Q: ", test_set["question"][200])
print("Correct answer:", correct_answer)

for i in range(100):
    print(f"RUN #{i+1}")
    text = few_shot_prompt(test_set["question"][i])
    print(text)
    answer = extract_answer(text)
    correct_answer = extract_correct_answer(train_set["answer"][i])
    print(f"  Model: {answer} | Expected: {correct_answer}")
    if answer == correct_answer:
        correct_count += 1

print("Final Correct count:", correct_count)
print("Accuracy:", correct_count / 100)

In [4]:
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset

# Load model and dataset once
model = SentenceTransformer("all-MiniLM-L6-v2")
gsm8k = load_dataset("gsm8k", "main", split="train")
questions = gsm8k["question"]
question_embeddings = model.encode(questions, convert_to_tensor=True, show_progress_bar=True)

def find_similar_questions(input_question: str, top_k: int = 3):
    query_embedding = model.encode(input_question, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, question_embeddings, top_k=top_k)[0] # cosine similarity
    return [
        {"score": h["score"], "question": questions[h["corpus_id"]], "answer": gsm8k[h["corpus_id"]]["answer"]}
        for h in hits
    ]

# Example
results = find_similar_questions("Sansa is a famous artist, she can draw a portrait and sell it according to its size. She sells an 8-inch portrait for $5, and a 16-inch portrait for twice the price of the 8-inch portrait. If she sells three 8-inch portraits and five 16-inch portraits per day, how many does she earns every 3 days?")
for r in results:
    print(f"[{r['score']:.3f}] {r['question']}\n")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/234 [00:00<?, ?it/s]

[1.000] Sansa is a famous artist, she can draw a portrait and sell it according to its size. She sells an 8-inch portrait for $5, and a 16-inch portrait for twice the price of the 8-inch portrait. If she sells three 8-inch portraits and five 16-inch portraits per day, how many does she earns every 3 days?

[0.669] An artist spends 30 hours every week painting. If it takes her 3 hours to complete a painting, how many paintings can she make in four weeks?

[0.665] Three years ago, Rosie purchased an art piece for $4000. The same art piece will be three times as much in another three years. How much money will the art piece have increased by?

